In [ ]:
import ast
import random

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.decomposition import PCA
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("linkedin_description_vectors.csv")

df.head(5)

In [ ]:
df["industry"].nunique()

In [ ]:
df.shape

In [ ]:
df = df.dropna(subset=["salary_min", "salary_max"])

df["salary"] = (df["salary_min"] + df["salary_max"]) / 2
print(df["salary"].describe())

## Without description

In [ ]:
selected_features = [
    "city",
    "seniority_level", 
    "job_type", 
    "job_function",
    "industry",
]

first_df = df[selected_features + ["salary"]].copy()

X = pd.get_dummies(first_df[selected_features], drop_first=True)
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)
y = df["salary"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    keras.layers.Dense(64, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    keras.layers.Dense(32, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    keras.layers.Dense(1)
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=100,
    restore_best_weights=True,
    verbose=1,
)

checkpoint = keras.callbacks.ModelCheckpoint(
    "models/first.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=5000,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
model.load_weights("models/first.keras")

In [ ]:
y_pred = model.predict(X_test_scaled).flatten()

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")

In [ ]:
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Val MAE')
plt.legend()
plt.title('MAE Over Epochs')
plt.show()

plt.plot(history.history['loss'], label='Train MSE')
plt.plot(history.history['val_loss'], label='Val MSE')
plt.legend()
plt.title('Loss Over Epochs')
plt.show()

In [ ]:
test_index = random.choice(X_test.index.tolist())
print("Testing on index:", test_index)

single_X = X_test.loc[[test_index]]
single_y_true = y_test.loc[test_index]

single_X_scaled = scaler.transform(single_X)

single_y_pred = model.predict(single_X_scaled).flatten()[0]

print(f"True salary:     ${single_y_true:,.2f}")
print(f"Predicted salary: ${single_y_pred:,.2f}")

## With description

In [ ]:
selected_features = [
    "city",
    "seniority_level", 
    "job_type", 
    "job_function",
    "industry",
    "description",
]

second_df = df[selected_features + ["salary"]].copy()

desc_df = pd.DataFrame(second_df["description"].tolist(), index=second_df.index)

def process_row(row):
    row = row.strip("[").strip("]").strip("\n")
    float_values = []
    i = 0
    while len(row) > 0:
        space_index = row.find(" ")
        while space_index == 0:
            row = row[1:]
            space_index = row.find(" ")
        if space_index == -1:
            if row == "": 
                break
            float_values.append(float(row))
            break
        value_str = row[:space_index]
        float_values.append(float(value_str))
        row = row[space_index + 1:]
        i += 1
        
    return float_values

new_df = np.empty((desc_df.shape[0], len(process_row(desc_df.iloc[0, 0]))))
for i in range(new_df.shape[0]):
    new_df[i, :] = process_row(desc_df.iloc[i, 0])
desc_df = pd.DataFrame(new_df, index=second_df.index)
desc_df = desc_df.add_prefix("desc_")

categorical_features = [f for f in selected_features if f != "description"]
cat_df = pd.get_dummies(second_df[categorical_features], drop_first=True)

cat_df = cat_df.apply(pd.to_numeric, errors="coerce").fillna(0)
desc_df = desc_df.apply(pd.to_numeric, errors="coerce").fillna(0)

X = pd.concat([cat_df, desc_df], axis=1)
y = second_df["salary"]

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    keras.layers.Dense(256, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    keras.layers.Dense(128, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    keras.layers.Dense(64, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    keras.layers.Dense(1)
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

In [ ]:
checkpoint = keras.callbacks.ModelCheckpoint(
    "models/second.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=5000,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
model.load_weights("models/second.keras")

In [ ]:
y_pred = model.predict(X_test_scaled).flatten()

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")

In [ ]:
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Val MAE')
plt.legend()
plt.title('MAE Over Epochs')
plt.show()

plt.plot(history.history['loss'], label='Train MSE')
plt.plot(history.history['val_loss'], label='Val MSE')
plt.legend()
plt.title('Loss Over Epochs')
plt.show()

In [ ]:
test_index = random.choice(X_test.index.tolist())
print("Testing on index:", test_index)

single_X = X_test.loc[[test_index]]
single_y_true = y_test.loc[test_index]

single_X_scaled = scaler.transform(single_X)

single_y_pred = model.predict(single_X_scaled).flatten()[0]

print(f"True salary:     ${single_y_true:,.2f}")
print(f"Predicted salary: ${single_y_pred:,.2f}")

## With reduced description

In [ ]:
selected_features = [
    "city",
    "seniority_level", 
    "job_type", 
    "job_function",
    "industry",
    "description",
]

third_df = df[selected_features + ["salary"]].copy()

desc_df = pd.DataFrame(second_df["description"].tolist(), index=second_df.index)

def process_row(row):
    row = row.strip("[").strip("]").strip("\n")
    float_values = []
    i = 0
    while len(row) > 0:
        space_index = row.find(" ")
        while space_index == 0:
            row = row[1:]
            space_index = row.find(" ")
        if space_index == -1:
            if row == "": 
                break
            float_values.append(float(row))
            break
        value_str = row[:space_index]
        float_values.append(float(value_str))
        row = row[space_index + 1:]
        i += 1
        
    return float_values

new_df = np.empty((desc_df.shape[0], len(process_row(desc_df.iloc[0, 0]))))
for i in range(new_df.shape[0]):
    new_df[i, :] = process_row(desc_df.iloc[i, 0])
desc_df = pd.DataFrame(new_df, index=second_df.index)
desc_df = desc_df.add_prefix("desc_")
pca = PCA(n_components=25)
desc_pca = pca.fit_transform(desc_df)
desc_pca = pd.DataFrame(desc_pca, index=df.index).add_prefix('pca_desc_')

categorical_features = [f for f in selected_features if f != "description"]
cat_df = pd.get_dummies(second_df[categorical_features], drop_first=True)

cat_df = cat_df.apply(pd.to_numeric, errors="coerce").fillna(0)
desc_pca = desc_pca.apply(pd.to_numeric, errors="coerce").fillna(0)

X = pd.concat([cat_df, desc_pca], axis=1)
y = second_df["salary"]

X.head(5)

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    keras.layers.Dense(512, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    keras.layers.Dense(256, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    keras.layers.Dense(128, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    keras.layers.Dense(64, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    keras.layers.Dense(1)
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

In [ ]:
checkpoint = keras.callbacks.ModelCheckpoint(
    "models/third.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=5000,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
model.load_weights("models/third.keras")

In [ ]:
y_pred = model.predict(X_test_scaled).flatten()

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")

In [ ]:
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Val MAE')
plt.legend()
plt.title('MAE Over Epochs')
plt.show()

plt.plot(history.history['loss'], label='Train MSE')
plt.plot(history.history['val_loss'], label='Val MSE')
plt.legend()
plt.title('Loss Over Epochs')
plt.show()

In [ ]:
test_index = random.choice(X_test.index.tolist())
print("Testing on index:", test_index)

single_X = X_test.loc[[test_index]]
single_y_true = y_test.loc[test_index]

single_X_scaled = scaler.transform(single_X)

single_y_pred = model.predict(single_X_scaled).flatten()[0]

print(f"True salary:     ${single_y_true:,.2f}")
print(f"Predicted salary: ${single_y_pred:,.2f}")